In [ ]:
import os
import re
from collections import defaultdict
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import Dataset
from transformers import AutoTokenizer, PreTrainedTokenizerFast
from tqdm import tqdm

#NAME_ ATTRIBUTION / SUBTRACTION ANALYSIS
#(tokenize with offsets, charge each token to the word it overlaps, split proportionally
#across word boundaries, keep whitespace-only tokens separate) so we can quote a
#"subtraction" gain figure (remove name_ placeholder savings from both gain and baseline)

load_dotenv("key.env")
login(os.getenv("HF_TOKEN"))

model = "mistralai/Mistral-7B-v0.1"
variant = "conversation"
column = "clean_conversation"
model_name = model.split("/")[-1]

NAME_PATTERN = re.compile(r"^name_\d+$", re.IGNORECASE)
WORD_PATTERN = re.compile(r"\S+") #a word is a run of non-whitespace characters, as tadanobu defined it

In [ ]:
test_corpus = Dataset.from_parquet("test_randomsplit.parquet")
original_tokenizer = AutoTokenizer.from_pretrained(model, use_fast=True)
retrained_tokenizer = PreTrainedTokenizerFast.from_pretrained(
    f"retrained_conversational_tokenizers/tokenizer_{model_name}_{variant}"
)
print("test conversations:", len(test_corpus))

In [ ]:
#for one text, returns: total tokens with each tokenizer, tokens spent on whitespace-only
#tokens, and a dict {word_text: tokens} charging each token to the word(s) it overlaps
def tokens_per_word(text, tokenizer):
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]
    words = [(m.group(0), m.start(), m.end()) for m in WORD_PATTERN.finditer(text)]

    word_tokens = defaultdict(float)
    whitespace_tokens = 0
    wi = 0 #index into words, advanced as we sweep left to right (both offsets and words are in order)
    for (start, end) in offsets:
        if start == end: #special/empty tokens have (0, 0) or similar
            continue
        #advance past words that end before this token starts
        while wi < len(words) and words[wi][2] <= start:
            wi += 1
        #collect every word this token's span overlaps (usually 0 or 1, occasionally 2 if it straddles a boundary)
        overlaps = []
        j = wi
        while j < len(words) and words[j][1] < end:
            w_text, w_start, w_end = words[j]
            overlap = min(end, w_end) - max(start, w_start)
            if overlap > 0:
                overlaps.append((w_text, w_start, overlap))
            j += 1
        if not overlaps:
            whitespace_tokens += 1 #token lies entirely in whitespace, belongs to no word
            continue
        total_overlap = sum(o[2] for o in overlaps)
        for w_text, w_start, overlap in overlaps:
            #split the token's weight across the words it touches, proportional to characters contributed
            word_tokens[(w_text, w_start)] += overlap / total_overlap

    return len(offsets), whitespace_tokens, word_tokens

In [ ]:
total_old = total_new = 0
whitespace_old = whitespace_new = 0
name_saving = 0.0
word_saving = 0.0

for text in tqdm(test_corpus[column], desc=f"attribution for {model_name}/{variant}"):
    if not text:
        continue
    old_total, old_ws, old_words = tokens_per_word(text, original_tokenizer)
    new_total, new_ws, new_words = tokens_per_word(text, retrained_tokenizer)

    total_old += old_total
    total_new += new_total
    whitespace_old += old_ws
    whitespace_new += new_ws

    #every word key present in either tokenizer's charge dict (usually the same set, since
    #both tokenizers see the same words - just in case, union of both)
    for key in set(old_words) | set(new_words):
        w_text = key[0]
        saving = old_words.get(key, 0.0) - new_words.get(key, 0.0)
        if NAME_PATTERN.match(w_text):
            name_saving += saving
        else:
            word_saving += saving

In [ ]:
total_saving = total_old - total_new
whitespace_saving = whitespace_old - whitespace_new
word_charged_saving = word_saving + name_saving #word_saving accumulates non-NAME_ words only; NAME_ is a separate accumulator, not a subset of it
gain_raw = 100 * total_saving / total_old

print(f"{model_name} / {variant}")
print(f"total_old={total_old}  total_new={total_new}  saving={total_saving}  gain={gain_raw:.2f}%")
print(f"whitespace-only saving: {whitespace_saving:.0f}  ({100*whitespace_saving/total_saving:.1f}% of saving)")
print(f"word-charged saving:    {word_charged_saving:.0f}  ({100*word_charged_saving/total_saving:.1f}% of saving)")
print(f"  of which NAME_ placeholders: {name_saving:.0f}  ({100*name_saving/total_saving:.1f}% of total saving, {100*name_saving/word_charged_saving:.1f}% of word-charged saving)")

#subtraction: remove the name_ savings from both numerator (saving) and denominator (baseline total)
gain_subtraction = 100 * (total_saving - name_saving) / (total_old - name_saving)
print(f"\ngain after subtracting NAME_ (subtraction convention): {gain_subtraction:.2f}%")
print(f"(raw gain was {gain_raw:.2f}%)")

In [ ]:
#save results, appending a row per (model, variant) run so repeated runs accumulate in one file
import csv

csv_filename = "name_attribution_results.csv"
fieldnames = [
    "model", "variant", "total_old", "total_new", "gain_raw",
    "whitespace_saving", "whitespace_saving_pct",
    "word_charged_saving", "word_charged_saving_pct",
    "name_saving", "name_saving_pct_of_total", "name_saving_pct_of_word_charged",
    "gain_subtraction",
]
file_exists = os.path.isfile(csv_filename)
with open(csv_filename, mode="a" if file_exists else "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    if not file_exists:
        writer.writeheader()
    writer.writerow({
        "model": model_name, "variant": variant,
        "total_old": total_old, "total_new": total_new, "gain_raw": round(gain_raw, 2),
        "whitespace_saving": round(whitespace_saving), "whitespace_saving_pct": round(100*whitespace_saving/total_saving, 1),
        "word_charged_saving": round(word_charged_saving), "word_charged_saving_pct": round(100*word_charged_saving/total_saving, 1),
        "name_saving": round(name_saving), "name_saving_pct_of_total": round(100*name_saving/total_saving, 1),
        "name_saving_pct_of_word_charged": round(100*name_saving/word_charged_saving, 1),
        "gain_subtraction": round(gain_subtraction, 2),
    })
print(f"results saved in {csv_filename}")